# Data
- Number of motor vehicles driven in Denmark 
- Monthly data starting January 2018
- Training set : January 2018 to December 2023 
- Test set : January 2024 to December 2024 
- Variable of interest : total : number of vehicles registered in Denmark at given time 


# 1. Plot Data

Explain plot :

# 2. Linear Trend Model



# 3. OLS



In [ ]:
# ============================================
# 3.1 Estimate parameters using OLS
# ============================================

# Load the data
setwd("c:/Users/leahi/Documents/DTU/Time Series Analysis/assignments/Assignment1")
D <- read.csv("DST_BIL54.csv")

# Prepare the data (from read_data.R)
D$time <- as.POSIXct(paste0(D$time, "-01"), "%Y-%m-%d", tz = "UTC")
D$year <- 1900 + as.POSIXlt(D$time)$year + as.POSIXlt(D$time)$mon / 12
D$total <- as.numeric(D$total) / 1E6  # Convert to millions

# Remove any rows with NA values
D <- D[complete.cases(D), ]

# Split into training and test sets
teststart <- as.POSIXct("2024-01-01", tz = "UTC")
Dtrain <- D[D$time < teststart, ]
Dtest <- D[D$time >= teststart, ]

# Define the time variable x for training data
x <- Dtrain$year

# Output variable y (total vehicles in millions)
y <- Dtrain$total

# Check dimensions
cat("Number of training observations:", length(y), "\n")
cat("Any NA in y:", any(is.na(y)), "\n")

# Design matrix X for linear trend model: y = theta1 + theta2 * x
# Column 1: intercept (ones), Column 2: time variable x
X <- cbind(1, x)

# OLS estimate: theta_hat = (X'X)^(-1) X'y
thetahat <- solve(t(X) %*% X) %*% (t(X) %*% y)

cat("\nOLS Parameter Estimates:\n")
cat("theta1 (intercept):", thetahat[1], "\n")
cat("theta2 (slope):    ", thetahat[2], "\n")

# The OLS estimates are calculated using the normal equations:
# theta_hat = (X'X)^(-1) X'y
# This minimizes the sum of squared residuals: sum((y - X*theta)^2)

In [ ]:
# ============================================
# 3.2 Standard errors and plot
# ============================================

# Calculate residuals
yhat_train <- X %*% thetahat
residuals <- y - yhat_train

# Number of observations and parameters
n <- length(y)
p <- ncol(X)

# Estimate of variance (sigma^2)
sigma2_hat <- sum(residuals^2) / (n - p)
sigma_hat <- sqrt(sigma2_hat)

cat("Estimated residual standard deviation (sigma):", sigma_hat, "\n\n")

# Variance-covariance matrix of theta_hat: Var(theta_hat) = sigma^2 * (X'X)^(-1)
var_thetahat <- sigma2_hat * solve(t(X) %*% X)

# Standard errors are the square root of the diagonal elements
se_theta <- sqrt(diag(var_thetahat))

cat("Parameter Estimates with Standard Errors:\n")
cat("theta1 =", round(thetahat[1], 4), " ± ", round(se_theta[1], 4), "\n")
cat("theta2 =", round(thetahat[2], 4), " ± ", round(se_theta[2], 4), "\n")

# Plot: observations as points, estimated mean as line
plot(x, y, 
     pch = 16, col = "blue",
     xlab = "Year", 
     ylab = "Total vehicles (millions)",
     main = "Global Linear Trend Model - Training Data")
lines(x, yhat_train, col = "red", lwd = 2)
legend("topleft", 
       legend = c("Observations", "Estimated mean"), 
       col = c("blue", "red"), 
       pch = c(16, NA), lty = c(NA, 1), lwd = c(NA, 2))

In [ ]:
# ============================================
# 3.3 Forecast for test set (12 months ahead)
# ============================================

# Time variable for test set
x_test <- Dtest$year
y_test <- Dtest$total

# Design matrix for test set
X_test <- cbind(1, x_test)

# Predicted values
yhat_test <- X_test %*% thetahat

# Prediction variance: Var(y_new) = sigma^2 * (1 + x_new' (X'X)^(-1) x_new)
# For prediction intervals we need the variance of the prediction error
XtX_inv <- solve(t(X) %*% X)

# Calculate prediction variance for each test point
pred_var <- sapply(1:nrow(X_test), function(i) {
  x_new <- X_test[i, ]
  sigma2_hat * (1 + t(x_new) %*% XtX_inv %*% x_new)
})
pred_se <- sqrt(pred_var)

# 95% prediction intervals (using t-distribution with n-p degrees of freedom)
alpha <- 0.05
t_crit <- qt(1 - alpha/2, df = n - p)

lower <- yhat_test - t_crit * pred_se
upper <- yhat_test + t_crit * pred_se

# Create results table
results <- data.frame(
  Month = format(Dtest$time, "%Y-%b"),
  Observed = round(y_test, 4),
  Predicted = round(yhat_test, 4),
  Lower_95 = round(lower, 4),
  Upper_95 = round(upper, 4)
)

print("Forecast Table for 2024 (values in millions of vehicles):")
print(results)

# Plot forecast with prediction intervals
plot(c(x, x_test), c(y, y_test),
     type = "n",
     xlab = "Year", 
     ylab = "Total vehicles (millions)",
     main = "Forecast with 95% Prediction Intervals")

# Training data and fit
points(x, y, pch = 16, col = "blue")
lines(x, yhat_train, col = "red", lwd = 2)

# Test data
points(x_test, y_test, pch = 17, col = "darkgreen", cex = 1.2)

# Forecast line and prediction intervals
lines(x_test, yhat_test, col = "red", lwd = 2)
lines(x_test, lower, col = "red", lty = 2)
lines(x_test, upper, col = "red", lty = 2)

# Shade the prediction interval
polygon(c(x_test, rev(x_test)), c(lower, rev(upper)), 
        col = rgb(1, 0, 0, 0.2), border = NA)

legend("topleft", 
       legend = c("Training data", "Test data", "Fitted/Forecast", "95% PI"), 
       col = c("blue", "darkgreen", "red", "red"), 
       pch = c(16, 17, NA, NA), 
       lty = c(NA, NA, 1, 2), lwd = c(NA, NA, 2, 1))

In [ ]:
# 3.5 : Residuals analysis to check OLS assumptions

###############################################################
# 3.5 Residual analysis
###############################################################
# OLS, assumptions are same variance 
# and be mutually uncorrelated
# We need to run tests to check whether this is correct

# 1. Plot residuals vs time
par(mfrow = c(2, 2))

plot(x, residuals, 
     pch = 16, col = "blue",
     xlab = "Year", ylab = "Residuals",
     main = "Residuals vs Time")
abline(h = 0, col = "red", lwd = 2)

# 2. Plot residuals vs fitted values (check for constant variance)
plot(yhat_train, residuals,
     pch = 16, col = "blue",
     xlab = "Fitted values", ylab = "Residuals",
     main = "Residuals vs Fitted Values")
abline(h = 0, col = "red", lwd = 2)

# 3. Histogram of residuals (check for normality)
hist(residuals, breaks = 15, col = "lightblue",
     main = "Histogram of Residuals",
     xlab = "Residuals")

# 4. QQ-plot (check for normality)
qqnorm(residuals, pch = 16, col = "blue")
qqline(residuals, col = "red", lwd = 2)

par(mfrow = c(1, 1))

# 5. Autocorrelation function (ACF) - check for uncorrelated residuals
acf(residuals, main = "ACF of Residuals", lag.max = 20)

# 6. Summary statistics
cat("\nResidual Summary Statistics:\n")
cat("Mean of residuals:", mean(residuals), "(should be ~0)\n")
cat("Std dev of residuals:", sd(residuals), "\n")



In [ ]:

cat("============================================================\n")
cat("LOCAL MODEL (WLS) - Variance-Covariance Matrix:\n")
cat("============================================================\n")
cat("Sigma_local = sigma^2 * W^(-1) where W = diag(lambda^(N-i))\n")
cat("- Observations are still uncorrelated (off-diagonal = 0)\n")
cat("- But observations have UNEQUAL variances\n")
cat("- Older observations have LARGER variance (less trusted)\n")
cat("- Newer observations have SMALLER variance (more trusted)\n\n")

cat("Diagonal elements (variances) of Sigma_local:\n")
cat("- Oldest observation (i=1):   Var =", round(Sigma_local[1,1], 4), "\n")
cat("- Middle observation (i=36):  Var =", round(Sigma_local[36,36], 4), "\n")
cat("- Newest observation (i=72):  Var =", round(Sigma_local[N,N], 4), "\n\n")

